# WRF - rain shifts time series txtfile creator
## wrf mat files (after processing) -> adjusted SWMM timeserie txtfiles


## Imports and files

In [1]:
import os
import pickle
import scipy.io as sio
import itertools
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import geopandas as gpd

import contextily as ctx
from shapely.geometry import Polygon, Point

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from rasterio.transform import from_origin
from collections import namedtuple

from swmm_api.input_file import read_inp_file, SwmmInput, section_labels as sections
from swmm_api import read_out_file, swmm5_run

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer

import h5py


## Functions

In [2]:
def wrf_array_to_gdf(rain_array, xll, yll, cellsize):
    """
    Convert a 2D array representing rainfall data into a GeoDataFrame with polygon geometries.

    Parameters:
    - rain_array (numpy.ndarray): 2D array representing rainfall data.
    - xll (float): X-coordinate of the lower left corner of the grid.
    - yll (float): Y-coordinate of the lower left corner of the grid.
    - cellsize (float): Size of each cell in the grid.

    Returns:
    - wrf_gdf (geopandas.GeoDataFrame): GeoDataFrame containing polygon geometries representing rainfall cells.

    This function takes a 2D array representing rainfall data and converts it into a GeoDataFrame with polygon geometries.
    Each cell in the array represents a rainfall value, and the function creates a polygon for each cell with its value
    as an attribute. The coordinate reference system is set to ITM (Israel Transverse Mercator).

    Note: The function adjusts the y-coordinate to handle the reversed north values. In the ITM coordinate system, the
    y-coordinate increases as you move south. Therefore, to align with the common representation of north-oriented grids,
    the function adjusts the y-coordinate calculation to ensure that the polygons are correctly positioned.
    """
    # Convert rainfall data to mm/hour
    rain_array = rain_array 
    
    # Calculate grid dimensions
    nrows, ncols = rain_array.shape
    # Define transformation parameters
    transform = from_origin(xll, yll, cellsize, cellsize)
    # Initialize a list to store geometries
    geometries = []
    # Iterate over rows
    for i in range(nrows):
        # Calculate y-coordinate

        y_coord = yll + i * cellsize

        # Iterate over columns
        for j in range(ncols):
            # Get rain value
            rain_value = rain_array[i, j]
            # Calculate the coordinates of the cell
            x_min = xll + j * cellsize
            x_max = x_min + cellsize
            y_min = y_coord
            y_max = y_min - cellsize
            # Create a polygon geometry
            polygon = Polygon([(x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max)])
            # Append to the list of geometries
            geometries.append((rain_value, polygon))
    # Create a GeoDataFrame
    wrf_gdf = gpd.GeoDataFrame(geometries, columns=['rain_value', 'geometry'])
    # Replace NaN with 0
    wrf_gdf['rain_value'].fillna(0, inplace=True)
    # Set the coordinate reference system to ITM
    wrf_gdf.crs = 'epsg:2039'
    
    return wrf_gdf


In [3]:
def create_TimeSeriesData_txt(basins_rain_df):
    """
    Update the TimeSeriesData values of SWMM inp file by using basins_rain_df.

    Args:
        basins_rain_df (DataFrame): DataFrame where each row represents a basin, and each column represents a timestep. 
                                    The values are rain (mm) for each basin in a timestep.
    Returns:
        str: Text representation of the updated timeseries data in the specified format.
    """
    # Initialize an empty string to store the text
    timeseries_text = ''
    # Write the header
    timeseries_text += ";;Name                 Date          Time         Value     \n"
    timeseries_text += ";;------------ ----------------- ------------- -------------\n"

    # Iterate over each row in the DataFrame
    for index, row in basins_rain_df.iterrows():
        # Get the basin name
        basin_name = int(row['Basin_name'])

        # Get the first timestamp to use it as the time series name
        first_timestamp = basins_rain_df.columns[1]  # the first timestamp column
        ts_name = first_timestamp.strftime('%Y%m%d') + f"_S{basin_name}"

        # Write the data to the text
        timestamps = basins_rain_df.columns[1:]  # excluding the first column 'Basin_name'
        values = row.values[1:]  # excluding the first column 'Basin_name'
        for ts, value in zip(timestamps, values):
            date_str = ts.strftime('%m/%d/%Y')
            time_str = ts.strftime('%H:%M:%S')
            timeseries_text += f"  {ts_name}      {date_str}    {time_str}   {value:.4f}\n"
        timeseries_text += ";;------------ ----------------- ------------- -------------\n"
    
    return timeseries_text


In [4]:
def process_wrf_data(rain_array_3d, raanana_basins_gdf, metadata, time_vector, wrf_basin_legend_gdf):

    time_steps = rain_array_3d.shape[0]

    time_diff = time_vector[1:] - time_vector[:-1]
    is_10_min_interval = (time_diff == pd.Timedelta(minutes=10)).all()

    if not is_10_min_interval:
        raise ValueError("Time differences are not exactly 10 minutes. Please check the time vector.")

    wrf_gdf_list = []

    for time in range(time_steps):
        rain_array_2d = rain_array_3d[time, :, :]
        wrf_gdf = wrf_array_to_gdf(rain_array_2d, **metadata)
        wrf_intersecting_basins = gpd.sjoin(wrf_gdf, raanana_basins_gdf, how="inner", predicate="intersects")
        wrf_indices = wrf_intersecting_basins.index.unique()
        wrf_filtered_gdf = wrf_gdf.loc[wrf_indices]
        wrf_gdf_list.append(wrf_filtered_gdf['rain_value'])

    wrf_poly_grid_gdf = pd.concat(wrf_gdf_list, axis=1)
    wrf_poly_grid_gdf.columns = time_vector

    wrf_poly_grid_gdf['geometry'] = wrf_gdf['geometry']
    wrf_poly_grid_gdf = gpd.GeoDataFrame(wrf_poly_grid_gdf, crs='epsg:2039')
    wrf_poly_grid_gdf.index.name = 'wrf_cell_num'

    basin_rain_columns = ['Basin_name'] + list(wrf_poly_grid_gdf.columns[:])
    basins_rain_df = pd.DataFrame()
    for basin in range(1, len(raanana_basins_gdf) + 1):
        basin_df = wrf_basin_legend_gdf[wrf_basin_legend_gdf['Basin_name'] == basin][['Basin_name', 'pct', 'wrf_cell_num']]
        basin_wrf_cells_df = wrf_poly_grid_gdf.loc[basin_df['wrf_cell_num']]
        basin_wrf_cells_df = pd.merge(basin_df, basin_wrf_cells_df, on='wrf_cell_num')
        weighted_avgs = []
        for col in basin_rain_columns[1:-1]:
            weighted_avg = sum(basin_wrf_cells_df[col] * basin_wrf_cells_df['pct'])
            weighted_avgs.append(weighted_avg)
        new_row = [basin] + weighted_avgs
        df_basin = pd.DataFrame([new_row], columns=basin_rain_columns[:-1])
        basins_rain_df = pd.concat([basins_rain_df, df_basin])
    basins_rain_df = basins_rain_df.reset_index(drop=True)
    basins_rain_df.fillna(0, inplace=True)

    resampled_dfs = []
    date = basins_rain_df.columns[1].strftime('%Y%m%d')  # Format date as YYYYMMDD
    
    return basins_rain_df


# Load basin data

In [5]:
# Raanana sub-basin shapefile
raanana_basins_shapfile = r'D:\Development\RESEARCH\Raanana\gis\GIS\28_subcatchments\raanana_28_subcatchments.shp'
raanana_basins_gdf = gpd.read_file(raanana_basins_shapfile)  # sub-basins poly
raanana_basins_gdf.drop(columns=['Shape_Area', 'Area_km2', 'Area_ha','Area_m2'], inplace = True)

In [ ]:
wrf_time_type = 'future'
# Select the range of shifts
x_shifts = np.arange(-10000, 10001, 500) # Example range covering -1000 to 1000 in steps of 500
y_shifts = np.arange(-30000, 30001, 500)  # Example range covering -1000 to 1000 in steps of 500

# Select time index for the plot and to extract one general rain 2d array
time_i = 0
mat_path = r'\\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\matfiles\{}'.format(wrf_time_type)


# List all .mat files in the directory
mat_files = [file for file in os.listdir(mat_path) if file.endswith('.mat')]

for mat_file in mat_files:
    mat_file_path = os.path.join(mat_path, mat_file)
    filename = 'event_' +  os.path.basename(mat_file_path).split('.')[0][-2:]
    
    # Read the .mat file
    with h5py.File(mat_file_path, 'r') as file:
        # Extract data
        rain_rate_data = np.array(file['rainRatePGW'])   ## for future events change to 'rainRatePGW'
#         rain_total_data = np.array(file['totalRainPGW'])  ## for future events change to 'totalRainPGW'
        lat_data = np.array(file['lat'])
        lon_data = np.array(file['lon'])
        time_data = np.array(file['timesList']).flatten()
        time_vector = pd.to_datetime(time_data - 719529, unit='D').round('1min')
    
    ## clip a relevent domain from the array
    lon_domain_strat, lon_domain_end = 280, 310 
    lat_domain_strat, lat_domain_end = 270, 360

    rain_array_3d = rain_rate_data[:, lat_domain_strat:lat_domain_end, lon_domain_strat:lon_domain_end]
    rain_array_2d = rain_array_3d[time_i, :, :]  # Access only one time slice

    # Define the coordinate reference system transformation using pyproj.Transformer
    transformer = Transformer.from_crs("epsg:4326", "epsg:2039", always_xy=True)
    lon_transformed, lat_transformed = transformer.transform(lon_data, lat_data)

    # Ensure the transformed coordinates are correctly reshaped
    lon_transformed = (lon_transformed.reshape(lon_data.shape)).astype(int)
    lat_transformed = (lat_transformed.reshape(lat_data.shape)).astype(int)

    # Define the cell size and domain limits
    xll = lon_transformed.min() + (lon_domain_strat*1000)
    xhr = lon_transformed.min() + (lon_domain_end*1000)
    yll = lat_transformed.min() + (lat_domain_strat*1000)
    yhr = lat_transformed.min() + (lat_domain_end*1000)
    cellsize = 1000

    # Directory path for saving text files
    txtfile_path = os.path.join(r"\\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles", wrf_time_type, filename)
    os.makedirs(txtfile_path, exist_ok=True)

    metadata = {
        'xll': xll,
        'yll': yll,
        'cellsize': cellsize
    }

    for x_shift in x_shifts:
        for y_shift in y_shifts:
            
            shift_name = f"rain_shift_x{'plus' if x_shift >= 0 else 'minus'}_{abs(x_shift)}_y{'plus' if y_shift >= 0 else 'minus'}_{abs(y_shift)}.txt"
            file_path = os.path.join(txtfile_path, shift_name)

            if os.path.exists(file_path):
                print("File already exists, skipping:", file_path)
                continue
                
            shifted_xll = metadata['xll'] + x_shift
            shifted_yll = metadata['yll'] + y_shift

            shifted_metadata = {
                'xll': shifted_xll,
                'yll': shifted_yll,
                'cellsize': cellsize
            }

            # Convert the array to a GeoDataFrame
            wrf_gdf = wrf_array_to_gdf(rain_array_2d, **shifted_metadata)

            # Perform spatial join to find wrf cells intersecting with basins
            wrf_intersecting_basins = gpd.sjoin(wrf_gdf, raanana_basins_gdf, how="inner", predicate="intersects")

            # Create an empty DataFrame to store the results
            wrf_basin_legend_gdf = pd.DataFrame(columns=['Basin_name', 'wrf_cell_num', 'pct', 'basin_geometry', 'wrf_geometry'])

            # Iterate over each basin in raanana_basins_gdf
            for basin_index, basin_row in raanana_basins_gdf.iterrows():
                basin_name = basin_row['Name']
                basin_geometry = basin_row['geometry']
                intersecting_wrf_cells = wrf_intersecting_basins[wrf_intersecting_basins['index_right'] == basin_index]
                basin_area = basin_geometry.area

                # Iterate over each intersecting wrf cell
                for index, wrf_cell in intersecting_wrf_cells.iterrows():
                    wrf_cell_num = wrf_cell.name
                    wrf_geometry = wrf_cell['geometry']

                    if basin_geometry.is_valid and wrf_geometry.is_valid:
                        intersection = basin_geometry.intersection(wrf_geometry)

                        if intersection.is_valid:
                            pct = (intersection.area / basin_area)  # Normalize by basin area

                            wrf_basin_legend_gdf = pd.concat([
                                wrf_basin_legend_gdf, 
                                pd.DataFrame({
                                    'Basin_name': [basin_name],
                                    'wrf_cell_num': [wrf_cell_num],
                                    'pct': [pct],
                                    'basin_geometry': [basin_geometry],
                                    'wrf_geometry': [wrf_geometry]
                                })
                            ], ignore_index=True)

            basins_rain_df = process_wrf_data(rain_array_3d, raanana_basins_gdf, shifted_metadata, time_vector, wrf_basin_legend_gdf)
            transform_timeseries_text = create_TimeSeriesData_txt(basins_rain_df)

            shift_name = f"rain_shift_x{'plus' if x_shift >= 0 else 'minus'}_{abs(x_shift)}_y{'plus' if y_shift >= 0 else 'minus'}_{abs(y_shift)}.txt"
            file_path = os.path.join(txtfile_path, shift_name)
            with open(file_path, 'w') as file:
                file.write(transform_timeseries_text)

            print("Text data saved to:", file_path)

#             # Uncomment below to visualize
#             # Plotting code goes here
#                     # To visualaize un-comment it--->       
#             # Define min and max values for the color bar
#             vmin = 0  # minimum value
#             vmax = 150  # maximum value

#     #         Plot the Raanana sub-basins
#             fig, ax = plt.subplots()
#             raanana_basins_gdf.plot(ax=ax, color='none', edgecolor='black')

#             # Plot the wrf data
#             wrf_gdf.plot(column='rain_value', cmap='viridis', legend=True, ax=ax, alpha=0.5)

#             # Define x and y axis limits
#             x_min, x_max = xll, xhr  # Replace with your desired x-axis limits
#             y_min, y_max = yll, yhr  # Replace with your desired y-axis limits
#             ax.set_xlim([x_min, x_max])
#             ax.set_ylim([y_min, y_max])

#             # Add legend for wrf data
#             plt.legend(['Raanana Sub-basins', 'Rainfall Data'])

#             # Get the start and end times
#             time = time_vector[time_i].strftime("%Y-%m-%d %H:%M")


#             # Title with the time range
#             plt.title(f'Raanana Sub-basins with wrf Rainfall Data\n{time}')

#             # Add axis labels
#             plt.xlabel('Easting')
#             plt.ylabel('Northing')

#             # Show the plot
#             plt.show()


File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_30000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_29500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_29000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_28500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_28000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yminus_27500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\

File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_3500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_4000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_4500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_5000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_5500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_10000_yplus_6000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event

File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_22000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_21500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_21000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_20500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_20000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yminus_19500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future

File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_14000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_14500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_15000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_15500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_16000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9500_yplus_16500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event

File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_8000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_8500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_9000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_9500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_10000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\rain_shift_xminus_9000_yplus_10500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_01\

...

Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_20000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_19500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_19000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_18500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_18000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_17500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yminus_17000.txt
Text data saved to: \\vscifs.cc.hu

Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_10000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_10500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_11000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_11500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_12000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_12500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xminus_500_yplus_13000.txt
Text data saved to: \\vscifs.cc.huji.ac.i

File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_9500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_10000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_10500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_11000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_11500.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yplus_12000.txt
File already exists, skipping: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_0_yp

Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_22000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_21500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_21000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_20500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_20000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_19500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yminus_19000.txt
Text data saved to: \\vscifs.cc.huji.ac.i

Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_8500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_9000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_9500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_10000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_10500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_11000.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab\hydrolab\home\Raz\WRF\txtfiles\future\event_04\rain_shift_xplus_500_yplus_11500.txt
Text data saved to: \\vscifs.cc.huji.ac.il\hydrolab